---
title: Query via h5netcdf
---

# Query NISAR GCOV via h5netcdf (baseline)

For comparison, open the same NISAR granule the traditional way using
earthaccess and h5netcdf, and query the same 10 random points.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="tqdm")

## Open with earthaccess + h5netcdf

In [ ]:
import earthaccess
import xarray as xr

earthaccess.login()

query = earthaccess.DataGranules()
query.short_name("NISAR_L2_GCOV_BETA_V1")
query.params["attribute[]"] = "int,FRAME_NUMBER,77"
query.params["attribute[]"] = "int,TRACK_NUMBER,5"
results = query.get(1)

files = earthaccess.open(results)
ds = xr.open_dataset(
    files[0],
    engine="h5netcdf",
    group="science/LSAR/GCOV/grids/frequencyA",
)
ds

## Query the same 10 random points

In [ ]:
import numpy as np
import time

rng = np.random.default_rng(42)
ny, nx = ds.dims["y"], ds.dims["x"]
yi = rng.integers(0, ny, size=10)
xi = rng.integers(0, nx, size=10)

t0 = time.perf_counter()
values = ds["HHHH"].values[yi, xi]
elapsed = time.perf_counter() - t0

print(f"Queried 10 points in {elapsed:.2f}s")

## Results

In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    {
        "y_index": yi,
        "x_index": xi,
        "HHHH": values,
    }
)
results_df

## Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
data = ds["HHHH"].values
im = ax.imshow(data, cmap="viridis", aspect="equal")
ax.scatter(xi, yi, c="red", s=40, edgecolors="white", linewidths=0.5, zorder=5)
ax.set_title("NISAR GCOV HHHH covariance")
plt.colorbar(im, ax=ax, shrink=0.6)
fig.tight_layout()

Note: `earthaccess.open` streams the file through HTTPS via fsspec.
Accessing `.values` may transfer a large portion of the file depending
on the HDF5 chunking. Compare the wall-clock time with the Icechunk
notebook, which fetches only the specific chunks containing the 10
query points.